## Lab Overview: Machine Learning Explainability

Once we have built and evaluated a machine learning model, the next step is to understand why the model makes the predictions it does. This is the focus of machine learning explainability. Explainability techniques help us examine how a model uses the available features to arrive at its predictions and provide insight into the patterns it has learned from the data.

In this lab, explainability serves several important purposes:

| Function | Description |
|---|---|
| **Understand feature influence** | Identify which features contribute most to the model's predictions |
| **Understand individual predictions** | Examine why the model classified a particular observation as benign or attack |
| **Validate learned patterns** | Determine whether the model appears to rely on meaningful signals or unexpected relationships |
| **Build trust in the model** | Provide evidence that helps analysts understand and assess model behavior |

These activities help us answer important questions about our trained models, such as:

- Which DNS features are most important to the model?
- What features contribute most strongly to distinguishing attack from benign traffic?
- Does the model rely on features that we expected to be useful based on our EDA?
- Why did the model classify a particular observation as an attack or benign?
- Do different observations receive predictions for different reasons?
- Can explainability help us identify unexpected or potentially problematic model behavior?

Explainability can be examined at different levels. **Global explainability** focuses on the model as a whole, helping us understand which features generally influence its predictions. **Local explainability** focuses on individual observations, helping us understand why the model made a particular prediction for a specific sample.

In this lab, we will use techniques such as feature importance to examine the model globally and LIME (Local Interpretable Model-agnostic Explanations) to investigate individual predictions.

> **Collectively, these tools allow us to move from understanding what the model predicts and how well it performs to understanding how the model arrives at those predictions.**

## DNS Data Exfiltration Challenge Problem Overview

In this lab, your objective is to use explainability methods to understand the outputs of ML models trained on DNS traffic contained in the **CIC-Bell-DNS-EXF-2021** dataset. Developed in collaboration with Bell Canada Cyber Threat Intelligence (CTI), this dataset focuses specifically on identifying DNS-based data exfiltration and covert command-and-control (C2) tunneling—common techniques used by adversaries to stealthily extract sensitive information from enterprise networks. Because DNS is an essential service, it is typically allowed through firewalls, making it an attractive channel for cybercriminals to encode and exfiltrate data without raising alarms.

For more details on the dataset, visit: [https://www.unb.ca/cic/datasets/dns-exf-2021.html](https://www.unb.ca/cic/datasets/dns-exf-2021.html)

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Handle Colab environment
colab = False
if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = './drive/MyDrive/NETCOM_DSD_TRAINING/DNSLab/'
else:
    base_dir = './'

# Set file path and load data
exfil_file = 'Exfil_Data.csv'
file_path = os.path.join(base_dir, 'data', exfil_file)

# Load data
data = pd.read_csv(file_path)

### Feature Definitions

The data consist of the following features:

| Feature | Data Type | Description |
|---|---|---|
| `FQDN_count` | Integer | Total number of characters in the fully qualified domain name (FQDN). |
| `subdomain_length` | Integer | Number of characters in the subdomain. |
| `upper` | Integer | Number of uppercase characters. |
| `lower` | Integer | Number of lowercase characters. |
| `numeric` | Integer | Number of numeric characters. |
| `entropy` | Float | Shannon entropy of the query name, measuring randomness/diversity. |
| `special` | Integer | Number of special characters (e.g., `-`, `_`, `=`, spaces). |
| `labels` | Integer | Number of labels in the domain name (separated by periods). |
| `labels_max` | Integer | Maximum length of any individual domain label. |
| `labels_average` | Float | Average length of the domain labels. |
| `longest_word` | Float | Ratio of the longest meaningful word to the total domain length. |
| `sld` | String | Second-level domain (SLD). |
| `len` | Integer | Total length of the domain and subdomain. |
| `subdomain` | Boolean | Indicates whether the domain contains a subdomain. |
| `label` | String | Target class/label (`attack` or `benign`). |


### Outcome Definitions

The data contain two classes of outputs.

| Class Label | Description |
|---|---|
| `attack` | DNS queries associated with data exfiltration activity. |
| `benign` | Legitimate DNS queries not associated with data exfiltration activity. |

## 2. Machine Learning Analysis

As in the previous lab, we will apply machine learning techniques to detect malicious activity. Using features extracted from the DNS traffic, we will train and evaluate a supervised learning model to classify network behavior as either **benign** or **attack**.

All the steps in this section are identical from the previous lab.

#### 2.1. Data Preparation

Before applying explainability techniques, we need a trained model and data on which to examine its predictions. We will use the same basic preparation process from the previous modeling lab.

In this section, we will:

1. **Separate features and target.**  
   Define the feature variables (`X`) and target variable (`y`).

2. **Split the data.**  
   Divide the data into training and test sets. The training data will be used to fit the model, while the test data will provide observations for explanation.

3. **Encode the data.**  
   Convert categorical features and target labels into the numeric representations required by the model.

4. **Fit the feature encoder.**  
   Fit the encoder using the training data and apply the same transformation to the test data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce

# Step 1: Separate features and target
X = data.drop(columns=["label"])
y = data["label"]

# Step 2: Split into training and test sets
# Stratification preserves the class distribution in both sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Step 3: Encode the target labels
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Step 4: Fit the encoder using training data only
categorical_columns = ["sld"]
encoder = ce.TargetEncoder(cols=categorical_columns)

X_train_encoded = encoder.fit_transform(X_train, y_train_encoded)

# Apply the same transformation to the test data
X_test_encoded = encoder.transform(X_test)

# Ensure all features are numeric
X_train_encoded = X_train_encoded.apply(pd.to_numeric, errors="coerce")

X_test_encoded = X_test_encoded.apply(pd.to_numeric, errors="coerce")

print(f"Training samples: {len(X_train_encoded):,}")
print(f"Test samples:     {len(X_test_encoded):,}")
print(f"Number of features: {X_train_encoded.shape[1]}")

#### 2.2. Training the Model

We need a trained model before we can examine how it makes predictions. To maintain consistency with the previous lab, we will use a **Random Forest classifier**.

In this section, we will:

1. **Instantiate the classifier.**
2. **Train the model** using the prepared training data.
3. **Generate predictions** for the test data.

The trained model and its predictions will provide the basis for the explainability techniques in the remainder of this lab.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Step 1: Initialize the Random Forest classifier
rf_classifier = RandomForestClassifier(random_state=42)

# Step 2: Train the classifier
rf_classifier.fit(X_train_encoded, y_train_encoded)

## 3. Explainability

After training and evaluating our model, we can examine why it makes the predictions it does. Explainability techniques provide insight into how the model uses features and how those features contribute to its predictions.

We will examine explainability at the global and local levels:

#### 3.1. Global Explainability

Global explainability examines the model's behavior across the dataset as a whole. It helps us understand which features are most influential in the model's predictions and whether the model appears to rely on features that we identified as potentially useful during EDA.

In this section, we will visualize the variable importance derived from the Random Forest model. Variable importance scores indicate how useful each feature is in making predictions, with higher scores reflecting features that contribute more significantly to the model's decisions.

Visualizing variable importance helps us understand which features are driving the predictions, allowing for better interpretation and insights into the model's behavior.

We will use the `RandomForestClassifier`'s built-in method to extract the importance scores and then plot them for easier interpretation.

In [ ]:
# Extract class and feature names
class_names = list(label_encoder.classes_)
feature_names = list(X_train_encoded.columns)

# Get feature importances from trained Random Forest
importances = rf_classifier.feature_importances_

# Calculate variability in feature importance across ensemble of trees
std = np.std([tree.feature_importances_ for tree in rf_classifier.estimators_], axis=0)
forest_importances = pd.Series(importances, index=feature_names)

# Sort feature importances in descending order
forest_importances_sorted = forest_importances.sort_values(ascending=False)
sorted_labels_to_original = {label: i for i, label in enumerate(feature_names)}
std_sorted_indices = [sorted_labels_to_original[label] for label in forest_importances_sorted.index]
std_sorted = std[std_sorted_indices]

# Plot the feature importances with error bars
fig, ax = plt.subplots()
forest_importances_sorted.plot.bar(yerr=std_sorted, ax=ax)

#### 3.1.1. Assignment

Now that you have visualized the global feature importance, examine the results carefully.

1. **Identify the top features.** Which 3-5 features have the highest importance scores?
2. **Connect to EDA.** Do these top features align with the insights you gained during the Exploratory Data Analysis (EDA) in the previous lab? Why or why not?
3. **Consider the domain.** Based on your knowledge of DNS exfiltration, why do these specific features make sense as indicators of malicious activity?

#### 3.2. Local Explainability

Global feature importance tells us which features are important overall, but it does not explain why the model made a particular prediction for an individual observation.

In this section, we use the LIME (Local Interpretable Model-agnostic Explanations) tool to better understand the reasoning behind individual predictions made by our trained Random Forest model.

LIME helps explain *why* a specific prediction was made by approximating the complex model locally with a simpler, interpretable model. This is especially useful when working with black-box models like Random Forests, where understanding the logic behind each prediction is not straightforward.

When you run the cell below:

* A **random record** from the test set is selected.
* Our trained model makes a prediction on that record.
* LIME then analyzes the prediction and provides:

  * The **predicted class** and the model’s **confidence** in that prediction.
  * A **feature-level explanation** showing which features influenced the decision the most and in what direction (i.e., whether they pushed the prediction toward or away from the predicted class).

This process can be repeated by rerunning the cell, allowing you to explore explanations for different data points and gain insights into how the model behaves across a variety of samples.

In [ ]:
import lime
import lime.lime_tabular

# LIME explanation
class_names = list(label_encoder.classes_)
feature_names = list(X_train_encoded.columns)

# Use a sample of X_train_encoded for LIME to avoid memory issues
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_encoded.values, 
    feature_names=feature_names, 
    class_names=class_names, 
    discretize_continuous=False
)

# Select a random instance from the test set for explanation
i = np.random.randint(0, X_test_encoded.shape[0])
print(i)
exp = explainer.explain_instance(X_test_encoded.iloc[i].values, rf_classifier.predict_proba, num_features=10, top_labels=1)
exp.show_in_notebook(show_table=True, show_all=False)

#### 3.2.1. Assignment

Using the LIME explanations, investigate how the model behaves on different types of samples.

1. **Compare predictions.** Run the LIME explanation cell multiple times. Do you notice any patterns in which features are most influential for different records?
2. **Analyze a 'correct' prediction.** Find a record where the model correctly identifies an 'attack' and observe which features drove that decision. Does the explanation align with your intuition?
3. **Analyze a 'challenging' prediction.** Find a record where the model might have lower confidence or where the features seem ambiguous. How does LIME describe the reasoning in these cases?


### Assignment

Reflect on the entire machine learning pipeline (EDA, Modeling, and Explainability) and answer the following:

1. **The Value of Explainability.** How does being able to explain individual predictions (local) and overall feature importance (global) change your confidence in the model?
2. **Detecting Malicious Activity.** Based on both the model's performance and its explanations, how effective is this approach for detecting DNS-based data exfiltration?
3. **Future Improvements.** Given what you've learned about the features and the model's decision-making, what is one way you could improve the model (e.g., new features, different algorithms, or more data)?
